# **Introduction**

The Goal of this notebook is to show the usefulness of my **Automated Problem Solver** and prove that we can get a relatively good score in usual supervised machine learning problems with a few lines of codes.

For more information about this Problem Solver, you can refer to the following notebook (and if you found it useful, upvote it please!)

[**Automated Problem Solver**](https://www.kaggle.com/code/khashayarrahimi94/automated-problem-solver-house-price)

## **Note:**
This solution is just a guide, it can tell us the level of hardness of the problem.

I suggest it as a **Starting Point** and not the End. It is obvious that for a nice accuracy we should focus more and more on Exploratory Data Analysis and also Modeling.

In [ ]:
!pip install verstack

In [ ]:
import pandas as pd
import numpy as np
from numpy import mean
import lightgbm as lgb
from lightgbm import LGBMRegressor
from verstack import LGBMTuner
from matplotlib import pyplot as plt
from sklearn.preprocessing import PowerTransformer,StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.datasets import fetch_california_housing

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s3e1/train.csv')
test = pd.read_csv(r'/kaggle/input/playground-series-s3e1/test.csv')
submission = pd.read_csv(r'/kaggle/input/playground-series-s3e1/sample_submission.csv')

In [ ]:
House_sklearn, y = fetch_california_housing(return_X_y=True)
House_sklearn = pd.DataFrame(House_sklearn, columns=train.columns[1:-1])
House_sklearn['MedHouseVal'] = y
House_sklearn

In [ ]:
train.drop(['id'],axis=1,inplace=True)
Train = pd.concat([train,House_sklearn],axis=0).reset_index(drop=True)
Train

## Remove Outliers 

In [ ]:
clf = IsolationForest(contamination =0.05,max_samples=0.7 ,random_state=0).fit(Train)
OD = clf.predict(Train.values)
Outlier_rows = []
for i in range(Train.shape[0]):
    
    if OD[i] == -1:
        Outlier_rows.append(i)
Train = Train.drop(Outlier_rows)
Train = Train.reset_index(drop=True)
Train

In [ ]:
test.drop(['id'],axis=1,inplace=True)
All = pd.concat([Train,test],axis=0).reset_index(drop=True)

# These 4 columns do not have completely gaussian distribution 
#need_scale = ['Population','AveOccup','Latitude','Longitude']
#All[need_scale] = StandardScaler().fit_transform(All[need_scale])
All

## LGBMTuner
This module tune the model automatically. 
For getting Stable prediction, we run the LGBMTuner multiple times and return their mean as the final prediction.

In [ ]:
def stable_prediction(n_trails):
    
    predictions = pd.DataFrame(columns = [i for i in range(n_trails)])
    
    for trail in range(n_trails):
        
        train_prepared = All.head(Train.shape[0])
        test_prepared = All.tail(test.shape[0])

        X = train_prepared.values[:,:-1]
        Y = train_prepared.values[:,-1]
        
        # the only required argument
        tuner = LGBMTuner(metric = 'rmse',trials = 150,seed = 13) 

        #the tuner needs these datatype for X and Y
        X = pd.DataFrame(X)
        Y = pd.Series(Y)
        tuner.fit(X,Y)
        test_df = pd.DataFrame(test_prepared.values[:,:-1])
        predicted = tuner.predict(test_df)

        predictions[trail] = predicted
        
    Mean_Prediction = []
    
    for i in range(predictions.shape[0]):
        
        row = predictions.iloc[i].values.tolist()
        Mean = mean(row)
        Mean_Prediction.append(Mean)
    
    return Mean_Prediction,predictions

In [ ]:
Pre = stable_prediction(12)

In [ ]:
submission['MedHouseVal'] = Pre[0]
submission

In [ ]:
submission.to_csv('submission.csv', index=False)

## **Hope you enjoy this short notebook friends. If you found it useful, please Upvote it.**